In [ ]:
import os
import re
import nltk
import pm4py
import stanza
import requests
import unicodedata
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from itertools import groupby
from collections import Counter
from gensim.models import Word2Vec
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.algo.evaluation.precision import algorithm as precision_evaluator
from pm4py.algo.evaluation.replay_fitness import algorithm as fitness_evaluator

import warnings
warnings.filterwarnings('ignore')

In [ ]:
def percentage_sampling(dataset, percentage):
    
    # The function performs sampling on a dataset of sequences (e.g., activity traces) based on their frequency. 
    # It counts the occurrences of each sequence, sorts them by frequency in descending order, and calculates the 
    # cumulative frequency. The function then selects sequences that contribute to a specified percentage 
    # of the total occurrences.
    #
    # Input:
    # - dataset: A list of sequences.
    # - percentage: The percentage of total occurrences to sample. Default is 20%.  
    # Note: Pareto principle, also commonly referred to as the 80/20 rule, is an observation and principle that suggests 
    # that approximately 80% of the results or effects are typically generated by only 20% of the causes or inputs. In other 
    # words, a significant portion of the output is driven by a relatively small portion of the factors or efforts. 
    #
    # Output:
    # - variants_count: The number of sequences selected to meet the specified percentage.
    # - sorted_sequences: A list of sequences sorted by frequency in descending order.

    # Count occurrences of each sequence (e.g., activity trace) in the dataset
    sequence_counts = Counter(map(tuple, dataset))

    # Sort the sequences by frequency in descending order
    sorted_sequences = sorted(sequence_counts.items(), key=lambda x: x[1], reverse=True)  # type dictionary

    # Calculate the total frequency (occurrences) of all sequences in the dataset
    total_frequency = sum(sequence_counts.values())
    cumulative_frequency = 0

    # Loop through the sorted_sequences and calculate cumulative frequency
    # Keep track of the sequences contributing to the specified percentage of total occurrences
    result_sequences = [1 for sequence, frequency in sorted_sequences
                        if (
                            cumulative_frequency := cumulative_frequency + frequency) / total_frequency * 100 <= percentage]
    variants_count = sum(result_sequences)
    
    if variants_count < 1:
        variants_count = 1
        
    return variants_count, sorted_sequences



def discover_petri_net(log_df):
    
    # The function takes a DataFrame representing an event log and performs the following tasks:
    # 1. Prepares timestamps for the event log.
    # 2. Converts the DataFrame to a PM4Py event log.
    # 3. Discovers a Petri net using Integer Linear Programming (ILP) based on the event log.

    # Input:
    # - log_df: A DataFrame containing event log data.

    # Output:
    # - net: The discovered Petri net.
    # - im: The initial marking of the Petri net.
    # - fm: The final marking of the Petri net.
    # - log: The event log converted to a PM4Py event log.
    
    # Prepare the event log timestamp
    current_timestamp = pd.Timestamp.now()
    timestamps = [(current_timestamp + pd.Timedelta(minutes=i)).strftime("%Y%m%dT%H%M") for i in range(len(log_df))]
    log_df['time:timestamp'] = pd.to_datetime(timestamps)

    # Convert the DataFrame to PM4Py event log
    log = log_converter.apply(log_df)
    
    # Discover Petri net using ILP
    net, im, fm = pm4py.discover_petri_net_ilp(log_df, activity_key='concept:name', 
                                               case_id_key='case:concept:name', timestamp_key='time:timestamp')

    return net, im, fm, log



def compute_extended_cardoso(petri_net):
    
    # The function computes the Extended Cardoso index, which measures the complexity of a process model by analyzing 
    # complex structures such as XOR, OR, AND, etc., in a Petri net.

    # Input:
    # - petri_net: A Petri net representation to analyze.

    # Output:
    # - The number of places in the Petri net.
    # - The number of transitions in the Petri net.
    # - The number of arcs in the Petri net.
    # - The Extended Cardoso index, representing the count of distinct sets of target transitions for each output arc 
    #   of a place.

    extended_cardoso = 0

    # Loop through each place in the Petri net
    for place in petri_net.places:
        # For each place, get the output arcs (out_arcs) from the place to transitions
        # and create a set of distinct target transitions for each output arc
        # (represented as frozensets to make them hashable)
        complex_structures = {frozenset(arc.target for arc in place.out_arcs)}

        # Increment the extended_cardoso count by the number of distinct complex structures
        extended_cardoso += len(complex_structures)

    return len(petri_net.places), len(petri_net.transitions), len(petri_net.arcs), extended_cardoso



def calculate_fmeasure(log, net, im, fm):
    
    # The function calculates the fitness, precision, and F-measure of a Petri net model given an event log.
    # It uses fitness and precision evaluators from PM4Py to make these calculations.

    # Input:
    # - log: The event log data.
    # - net: The Petri net model.
    # - im: The initial marking of the Petri net.
    # - fm: The final marking of the Petri net.

    # Output:
    # - fitness: The fitness of the Petri net model on the event log.
    # - precision: The precision of the Petri net model on the event log.
    # - fmeasure: The F-measure, a measure of both fitness and precision.

    variant = fitness_evaluator.Variants.ALIGNMENT_BASED
    fitness = fitness_evaluator.apply(log, net, im, fm, variant=variant)['average_trace_fitness']

    variant = precision_evaluator.Variants.ALIGN_ETCONFORMANCE
    parameters = {variant.value.Parameters.SHOW_PROGRESS_BAR: False}
    precision = precision_evaluator.apply(log, net, im, fm, variant=variant, parameters=parameters)
    fmeasure = 2 * fitness * precision / (fitness + precision) if fitness != 0 else 0

    return fitness, precision, fmeasure

In [ ]:
event_log = pd.read_pickle("./eventlog.pkl")
current_timestamp = pd.Timestamp.now()
timestamps = [(current_timestamp + pd.Timedelta(minutes=i)).strftime("%Y%m%dT%H%M") for i in range(len(event_log))]
event_log['time:timestamp'] = pd.to_datetime(timestamps)
log = pm4py.convert_to_event_log(event_log)

variants = event_log.groupby('case:concept:name')['concept:name'].apply(list).reset_index()['concept:name']
results_list = []
for percentage in range(10, 91, 5):
    variant_count, sorted_traces = percentage_sampling(variants, percentage)
    new_log = pd.DataFrame([x for x in [[[str(index), x] for x in trace] for index, trace in
                                               enumerate([list(x[0]) for x in sorted_traces[:variant_count]])] for x in
                                   x], columns=['case:concept:name', 'concept:name'])
    new_net, new_im, new_fm, new_log = discover_petri_net(new_log)
    new_places, new_transitions, new_arcs, new_extended_cardoso = compute_extended_cardoso(new_net)
    new_fitness, new_precision, new_fmeasure = calculate_fmeasure(log, new_net, new_im, new_fm)
    
    result = {
        'percentage': percentage,
        'variant_count': variant_count,
        'variants': sorted_traces[:variant_count],
        'model size':  ' x '.join([str(new_places), str(new_transitions), str(new_arcs)]),
        'cardoso': new_extended_cardoso,
        'precision': new_precision,
        'fitness': new_fitness,
        'fmeasure': new_fmeasure,
        'initial_marking': new_im,
        'final_marking': new_fm,
        'Petri Net': new_net
    }
    print(f'A legal process model with {variant_count} trace variants, generated using the Sample Filtering ({percentage}%), and an f-measure of {new_fmeasure}')
    
    results_list.append(result)
    
    if percentage > 10:
        if results_list[-2]['fmeasure'] > new_fmeasure:
            break
    
# Convert the results_list to a dataframe
results_df = pd.DataFrame(results_list)

outdir = f"./Results"
if not os.path.exists(outdir):
    os.makedirs(outdir)
outname = 'results_df.pkl'
fullpath = os.path.join(outdir, outname)
results_df.to_pickle(fullpath)

In [ ]:
titles = [t.label for t in new_net.transitions if t.label is not None]

In [ ]:
titles = [t.label for t in new_net.transitions if t.label is not None]
def get_texts_for_title(event_log, title):
    return event_log[event_log['concept:name'] == title]['concept:name:section'].tolist()  

In [ ]:
def get_parag_for_title(event_log, title):
    return event_log[event_log['concept:name'] == title]['concept:name:paragraphs'].tolist()  

PROVA 0

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a legal assistant specialized in analyzing legal texts. 
    Your goal is to explain the general legal concept that emerges when analyzing the following legal texts grouped under the title: {title}. You are not summarizing each paragraph individually — instead, abstract the underlying legal theme that unites them all, even if they differ in specific facts or rulings.

    Avoid references to specific directives, articles, or case numbers. Focus only on what the texts collectively reveal about the legal principle, doctrine, or procedural rule they reflect.

    ### Input:
    The legal paragraphs related to {title} are as follows: {texts}

    ### Output format:
    “The concept that unites these paragraphs is as follows:

    [Provide a concise, structured explanation of the common legal theme.]

    However, there are variations in [briefly mention any recurring exceptions or deviations, without going into detail].”

    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_parag_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df3.to_csv(os.path.join(outdir, "summaries3.csv"), index=False)


In [ ]:
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

In [ ]:
texts

PROVA 1

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a legal text summary model. Your task is to analyze multiple legal paragraphs related to a given title and generate a **concise and structured summary** that focuses only on the common elements present in all paragraphs.  

    ### **Guidelines:**  
    - Identify **the main points shared** across the paragraphs.  
    - **Avoid case-specific details** and focus on general principles.  
    - Present the summary in a **clear and structured manner**.  
    - If there are variations, mention them briefly at the end, without detailing specific cases or articles.  

    ### **Input:**  
    The following are multiple legal paragraphs related to the title: {title}  

    {'. '.join(texts)}  

    ### **Output format:**  
    In these paragraphs, the common elements are as follows:  

    [Write the main shared aspects in a concise, structured, and formal manner.]  

    However, there are variations in [briefly mention differing aspects without going into details].  
    
    Write a **single, smooth paragraph** that summarizes the key points in a **clear and natural manner**. Focus on the **general legal principles**, avoiding excessive legal jargon, case numbers, or specific article references unless absolutely necessary.  

    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df3.to_csv(os.path.join(outdir, "summaries3.csv"), index=False)


In [ ]:
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 2

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a legal text summary model. Your task is to analyze multiple legal paragraphs related to a given title and generate a **concise and structured summary** that focuses only on the common elements present in all paragraphs.  

    ### **Guidelines:**  
    - Identify **the main points shared** across the paragraphs.  
    - **Avoid case-specific details** and focus on general principles.  
    - Present the summary in a **clear and structured manner**.  
    - If there are variations, mention them briefly at the end, without detailing specific cases or articles.  

    ### **Input:**  
    The following are multiple legal paragraphs related to the title: {title}  

    {'. '.join(texts)}  

    ### **Output format:**  
    In these paragraphs, the common elements are as follows:  

    [Write the main shared aspects in a concise, structured, and formal manner.]  

    However, there are variations in [briefly mention differing aspects without going into details].  

    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df.to_csv(os.path.join(outdir, "summaries.csv"), index=False)


In [ ]:
for index, row in summary_df.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 3

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a legal text summary model. Your task is to analyze multiple legal paragraphs related to a given title and generate a **highly generalized summary** that captures only the **core legal principles** common across all paragraphs.

    ### **Guidelines:**
    - Extract **only** the fundamental legal concepts and principles, **avoiding specific case details, country names, or article references**.
    - Focus on broad legal themes rather than procedural or contextual specifics.
    - Present the summary in a **clear and structured manner**, maintaining a neutral and formal tone.
    - If variations exist, summarize them at a high level **without referencing specific cases, articles, or regulations**.

    ### **Input:**
    The following are multiple legal paragraphs related to the title: {title}

    {'. '.join(texts)}

    ### **Output format:**
    In these paragraphs, the shared legal principles are as follows:
    
    [Write the main shared legal concepts in a **concise, abstract, and formal manner**.]

    However, some differences exist in [briefly outline general differences without naming specific cases, laws, or regions].

    Write a **cohesive and abstract summary** that encapsulates the **essential legal principles** while removing unnecessary procedural details, case names, or regulation numbers. Keep the language formal but accessible.

    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df.to_csv(os.path.join(outdir, "summaries.csv"), index=False)

In [ ]:
for index, row in summary_df.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 4

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a model who specializes in summarizing legal texts. Your task is to read the following paragraphs and summarize their **general principles**, avoiding technical details and specific regulatory references unless absolutely necessary.

    ### **Guidelines:**
    - Summarize **common concepts** clearly and naturally.
    - **Avoid references to individual articles of law or directive numbers** unless essential.
    - Write the summary in **a single, flowing paragraph**, without bullet points.
    - The language should be **formal but accessible**, reducing excessive jargon.

    ### **Texts to analyze:**
    {'. '.join(texts)}

    ### **General summary:**
    [Write the flowing, concise summary based on the common principles here]
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df3.to_csv(os.path.join(outdir, "summaries3.csv"), index=False)


In [ ]:
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 5

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a model specialized in summarizing legal texts. Your task is to extract and present the **general principles** from the following paragraphs, focusing on clarity and readability while avoiding excessive legal jargon.

    ### **Guidelines:**
    - Identify and summarize **common legal concepts** in a way that flows naturally, as if explaining them to a well-informed but non-specialist reader.
    - **Avoid specific references** to legal articles, directive numbers, case names, or numerical data unless strictly necessary to understand the core principle.
    - Present the summary as **a single, coherent paragraph**, ensuring a smooth, engaging, and accessible narrative.
    - Use **formal yet natural language**, simplifying complex legal expressions where possible while maintaining accuracy.

    ### **Texts to analyze:**
    {'. '.join(texts)}

    ### **General summary:**
    [Write the smooth, concise, and engaging summary focusing on common principles here]
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df3.to_csv(os.path.join(outdir, "summaries3.csv"), index=False)


In [ ]:
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 6

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are an expert in summarizing legal texts into accessible and concise explanations. Your task is to extract the general principles from the following legal texts, emphasizing broad concepts rather than technical details.

    Guidelines:
    Summarize in a single paragraph without technical jargon.

    Do not include specific article numbers or procedural details unless absolutely necessary.

    Focus on the core ideas and their impact rather than legal formalities.

    The tone should be clear, direct, and accessible to non-experts.

    Texts to analyze:
    {'. '.join(texts)}

    General summary:
    [Write the concise, general summary here]
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df3.to_csv(os.path.join(outdir, "summaries3.csv"), index=False)


In [ ]:
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 7

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a model specialized in summarizing legal texts into concise and accessible explanations. Your task is to extract the general principles from the following legal texts, emphasizing broad legal concepts over procedural details.

    Guidelines:
    Limit each summary to a maximum of 2-3 sentences.

    Avoid specific article numbers, directive names, case names, and procedural details unless absolutely necessary to understand the principle.

    Focus on general legal rules and implications, rather than individual cases or procedural aspects.
    
    The language should be clear, direct, and accessible to non-experts.

    Texts to analyze:
    {'. '.join(texts)}

    General summary:
    [Write a concise summary focusing only on broad legal principles and their implications.]
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df3.to_csv(os.path.join(outdir, "summaries3.csv"), index=False)


In [ ]:
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 8

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are an expert in summarizing legal texts into accessible and concise explanations. Your task is to extract the general principles from the following legal texts, emphasizing broad concepts rather than technical details.

    Guidelines:

    Summarize in a single, well-structured paragraph that flows naturally.

    Avoid technical jargon and excessive legal details.

    Do not list multiple points separately; instead, integrate them smoothly into a coherent narrative.

    Do not include specific article numbers, case names, or procedural references unless absolutely necessary for clarity.

    Focus on the core legal principles and their broader implications.

    The tone should be clear, direct, and accessible, suitable for a well-informed but non-specialist reader.

    Texts to analyze:
    {'. '.join(texts)}

    General summary:
    [Write the concise, clear, and engaging summary here]
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df3.to_csv(os.path.join(outdir, "summaries3.csv"), index=False)


In [ ]:
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 9

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a legal text summary model with expertise in analyzing legal documents. Your task is to analyze multiple legal paragraphs related to a given title and generate a **concise and structured summary** that focuses only on the common legal principles.  

    ### **Guidelines:**  
    - Identify **only the core legal principles** shared across all paragraphs.  
    - **Do NOT mention specific articles, case details, or judicial chambers.**  
    - Present the summary in a **neutral and structured manner**.  
    - If there are variations, mention them generally at the end, without specifics.  

    ### **Input:**  
    The following are multiple legal paragraphs related to the title: {title}  

    {texts}  
    
    Could you infer a general provision for the set of paragraphs?
    """

    # Use the "llama" model for processing
    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = paragraphs_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df3.to_csv(os.path.join(outdir, "summaries3.csv"), index=False)


In [ ]:
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 10

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a legal text summary model with expertise in analyzing legal documents. Your task is to analyze multiple legal paragraphs related to a given title and generate a concise and structured summary that focuses only on the core legal principles shared across all paragraphs.

    Guidelines:
    Identify only the general legal principles applicable across all the provided paragraphs.

    Avoid mentioning specific articles, case details, judicial chambers, or laws. Focus on the broader concepts that apply universally across different legal contexts.

    Use neutral language and structure the summary in a way that highlights the core principles without being tied to a specific jurisdiction or case.

    If there are variations in how the principles are applied, mention them in very broad terms, without referencing specific norms or cases.

    The summary should describe the principles in a way that makes them understandable in a variety of legal environments.

    Input:
    The following are multiple legal paragraphs related to the title: {title}

    {texts}

    Could you infer a general provision for the set of paragraphs??
    """

    # Use the "llama" model for processing
    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = paragraphs_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df3.to_csv(os.path.join(outdir, "summaries3.csv"), index=False)


In [ ]:
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 11

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a model who specializes in analyzing and summarizing legal texts. Your task is to analyze multiple legal paragraphs related to a given title and generate a **concise and structured summary** that focuses exclusively on common legal principles.

    ### **Guidelines:**
    - **Identify only the fundamental legal principles** that all paragraphs share.
    - **Avoid specific references to rules, articles of law, legal cases or judicial bodies.**
    - Present the summary in a **neutral and structured form**, highlighting key concepts and general principles.
    - If there are variations between paragraphs, mention them in a general way at the end, without specific details.
    - **Avoid interpretations or personal opinions**: the summary should reflect exclusively the content of the texts analyzed.
    - Use clear and precise language, suitable for a legal context.

    ### **Input:**
    Below are more legal paragraphs related to the title: **{title}**

    {texts}

    ### **Expected Output:**
    Please provide a clear and concise summary, no more than **5-7 sentences**, highlighting only the legal principles common to the texts provided. If a common principle cannot be identified, briefly explain why.
    """

    # Use the "llama" model for processing
    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = paragraphs_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df3.to_csv(os.path.join(outdir, "summaries3.csv"), index=False)


In [ ]:
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 12

In [ ]:
import ollama

def summarize_texts(texts, title):
    if not texts:
        return "Nessun testo disponibile per il riassunto."
    
    prompt = f"""
    You are a legal text summary model. Your task is to analyze multiple legal paragraphs related to a given title and generate a **clear and generalized summary** of the basic legal principles, avoiding references to specific laws, articles, or jurisdictions.
    
    ### **Guidelines:**  
    - Use simple and formal language.
    - The summary must be under 100 words.
    - Rephrase and restructure to ensure natural flow and coherence. 
    - **Avoid case-specific details** and focus on general principles.  
    - Present the summary in a **clear and structured manner**.  
    - If there are variations, mention them briefly at the end, without detailing specific cases or articles.  

    ### **Input:**  
    The following are multiple legal paragraphs related to the title: {title}  

    {'. '.join(texts)}  

    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    summary = summarize_texts(texts, title)
    summary_results[title] = summary

# Salvataggio dei risultati
import pandas as pd
import os

outdir = "."
if not os.path.exists(outdir):
    os.makedirs(outdir)

summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
summary_df3.to_csv(os.path.join(outdir, "summaries3.csv"), index=False)


In [ ]:
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

APPROCCIO IBRIDO

In [ ]:
def paragraphs_for_title(event_log, title):
    return event_log[event_log['concept:name'] == title]['concept:name:paragraphs'].tolist()  

In [ ]:
def associa_testi(event_log, title):
    return event_log[
        (event_log['concept:name'] == title) &  # Filtra per titolo
        (event_log['concept:name:section'].notna()) &  # Esclude NaN
        (event_log['concept:name:section'].str.strip() != "")  # Esclude stringhe vuote
    ]['concept:name:section'].tolist()


PROVA 1

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def extractive_summary(texts, num_sentences=3):
    if not texts:  # Controlla se la lista è vuota
        return "Nessun testo disponibile per il riassunto."

    sentences = " ".join(texts).split('. ')  # Concatena e poi tokenizza

    if not any(sentences):  # Controlla se la tokenizzazione ha prodotto solo stringhe vuote
        return "Testo non valido per il riassunto."

    vectorizer = TfidfVectorizer()
    try:
        tfidf_matrix = vectorizer.fit_transform(sentences)
    except ValueError:
        return "Il testo contiene solo stopwords o è vuoto."

    scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
    top_sentences = np.argsort(scores)[-num_sentences:]  # Prendi le più rilevanti
    return " ".join([sentences[i] for i in sorted(top_sentences)])


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    summary = extractive_summary(texts)
    summary_results[title] = summary

In [ ]:
import ollama

def summarize_texts(texts):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a legal text summary model with expertise in analyzing legal documents. Your task is to analyze multiple legal paragraphs related to a given title and generate a concise and structured summary that focuses only on the core legal principles shared across all paragraphs.

    Guidelines:
    
    Avoid mentioning specific articles, case details, judicial chambers, or laws. Focus on the broader concepts that apply universally across different legal contexts.

    Use neutral language and structure the summary in a way that highlights the core principles without being tied to a specific jurisdiction or case.

    If there are variations in how the principles are applied, mention them in very broad terms, without referencing specific norms or cases.

    The summary should describe the principles in a way that makes them understandable in a variety of legal environments.

    Input:
    {extractive}

    Could you infer a general provision for the set of paragraphs??
    """

    # Use the "llama" model for processing
    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    extractive = extractive_summary(texts)
    final_summary = summarize_texts(extractive)
    summary_results[title] = final_summary

In [ ]:
summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 2

In [ ]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

def extractive_summary(texts, num_sentences=3):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    sentences = [sent for text in texts for sent in sent_tokenize(text)]  # Tokenizza meglio le frasi

    if not sentences:
        return "Testo non valido per il riassunto."

    vectorizer = TfidfVectorizer()
    try:
        tfidf_matrix = vectorizer.fit_transform(sentences)
    except ValueError:
        return "Il testo contiene solo stopwords o è vuoto."

    scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
    top_sentences = np.argsort(scores)[-num_sentences:]  
    return " ".join([sentences[i] for i in sorted(top_sentences)])


In [ ]:
def summarize_texts(texts):
    if not texts or isinstance(texts, str) and "Nessun testo disponibile" in texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a legal text summary model with expertise in analyzing legal documents. Your task is to analyze multiple legal paragraphs related to a given title and generate a concise and structured summary that focuses only on the core legal principles shared across all paragraphs.

    Guidelines:
    
    - Avoid mentioning specific articles, case details, judicial chambers, or laws. 
    - Focus on the broader concepts that apply universally across different legal contexts.
    - If variations exist, describe them in general terms without references to specific norms or cases.

    Input:
    {texts}

    Could you infer a general provision for the set of paragraphs?
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    extractive = extractive_summary(texts)
    final_summary = summarize_texts(extractive)  # Ora passa un testo corretto
    summary_results[title] = final_summary

In [ ]:
summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 3

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Carica un modello di embedding semantico
model = SentenceTransformer('all-MiniLM-L6-v2')

def filter_relevant_texts(texts, title, threshold=0.3):
    """Filtra i testi selezionando solo quelli semanticamente vicini al titolo."""
    if not texts:
        return []

    title_embedding = model.encode(title, convert_to_tensor=True)
    text_embeddings = model.encode(texts, convert_to_tensor=True)
    
    # Calcola la similarità coseno tra il titolo e ogni paragrafo
    similarities = util.pytorch_cos_sim(title_embedding, text_embeddings)[0].cpu().numpy()
    
    # Seleziona i testi con similarità sopra la soglia
    relevant_texts = [texts[i] for i in range(len(texts)) if similarities[i] > threshold]
    
    return relevant_texts if relevant_texts else ["Nessun testo pertinente trovato."]


In [ ]:
def summarize_texts(texts, title):
    if not texts or "Nessun testo pertinente" in texts:
        return f"Nessun testo pertinente trovato per {title}."
    
    # Se abbiamo un solo paragrafo dopo il filtraggio, restituiamolo direttamente
    if len(texts) == 1:
        return f"Relevant legal principle: {texts[0]}"

    prompt = f"""
    You are a legal expert specializing in summarizing legal texts. Your task is to analyze multiple legal paragraphs related to a given title and extract a structured summary focusing on core legal principles.

    Guidelines:

    - Always extract a general legal principle, even if the text is vague or incomplete.
    - If the text lacks explicit legal provisions, infer a general concept that applies to the topic.
    - Avoid phrases like "no relevant text found"—instead, identify the closest legal principle.

    Input:
    {texts}

    What general legal principle can be inferred from these paragraphs?
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)  
    relevant_texts = filter_relevant_texts(texts, title)  # Filtra prima!
    final_summary = summarize_texts(relevant_texts, title)  # Poi genera il riassunto
    summary_results[title] = final_summary

In [ ]:
summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

In [ ]:
cost = get_texts_for_title(event_log, 'cost')
cost

PROVA 4

In [ ]:
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from collections import Counter

def extractive_summary(texts, num_sentences=5):
    """Seleziona le frasi chiave usando embedding e clustering."""
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = [sent for text in texts for sent in text.split('.')]
    embeddings = model.encode(sentences)
    
    # Clustering per trovare frasi rappresentative
    kmeans = KMeans(n_clusters=num_sentences, random_state=42)
    kmeans.fit(embeddings)
    selected_indices = []
    for cluster in set(kmeans.labels_):
        idx = np.where(kmeans.labels_ == cluster)[0][0]
        selected_indices.append(idx)
    
    return [sentences[i] for i in selected_indices]

def abstractive_summary(texts):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    prompt = f"""
    You are a legal text summary model. Your task is to analyze multiple legal paragraphs related to a given title and generate a **concise and structured summary** that focuses only on the common elements present in all paragraphs.  

    ### **Guidelines:**  
    - Identify **the main points shared** across the paragraphs.  
    - **Avoid case-specific details** and focus on general principles.  
    - Present the summary in a **clear and structured manner**.  
    - If there are variations, mention them briefly at the end, without detailing specific cases or articles.  

    ### **Output format:**  
    In these paragraphs, the common elements are as follows:  

    [Write the main shared aspects in a concise, structured, and formal manner.]  

    However, there are variations in [briefly mention differing aspects without going into details].  
    
    Write a **single, smooth paragraph** that summarizes the key points in a **clear and natural manner**. Focus on the **general legal principles**, avoiding excessive legal jargon, case numbers, or specific article references unless absolutely necessary.  

    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

def summarize_legal_texts(texts):
    extracted = extractive_summary(texts)
    combined_text = ' '.join(extracted)
    abstract_summary = abstractive_summary(combined_text)
    return abstract_summary


summary = summarize_legal_texts(cost)
print(summary)


PROVA 5

In [ ]:
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Carica il modello NLP per segmentare il testo in frasi
nlp = spacy.load("en_core_web_sm")

def split_sentences(texts):
    """Segmenta i paragrafi in frasi utilizzando spaCy."""
    sentences = []
    for text in texts:
        doc = nlp(text)
        sentences.extend([sent.text.strip() for sent in doc.sents])
    return sentences

def optimal_k(embeddings):
    """Determina il numero ottimale di cluster usando il metodo Elbow."""
    distortions = []
    max_clusters = min(10, len(embeddings))  # Non più cluster di quanti siano i dati
    for k in range(2, max_clusters + 1):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)
        distortions.append(kmeans.inertia_)
    return distortions.index(min(distortions)) + 2  # Il punto di minima variazione

def extractive_summary(texts):
    """Seleziona frasi chiave con embeddings e clustering ottimizzato."""
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = split_sentences(texts)
    if len(sentences) <= 5:
        return sentences  # Se il testo è breve, restituisci tutte le frasi

    embeddings = model.encode(sentences)
    k = optimal_k(embeddings)  # Determina il numero ottimale di frasi da estrarre
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)

    # Seleziona frasi più vicine al centroide generale
    centroid = np.mean(embeddings, axis=0)
    similarities = cosine_similarity([centroid], embeddings)[0]
    top_indices = np.argsort(similarities)[-k:]  # Prendi le frasi più simili al centroide
    selected_sentences = [sentences[i] for i in sorted(top_indices)]
    
    return selected_sentences

def abstractive_summary(title, texts):
    """Genera un riassunto astrattivo con Mistral basato sul titolo specifico."""
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    extracted_text = ' '.join(extractive_summary(texts))

    prompt = f"""
    You are a legal text summarization model. Your task is to analyze multiple legal paragraphs related to the title "{title}" and generate a **concise and structured summary** that focuses only on the common elements present in all paragraphs.  

    ### **Guidelines:**  
    - Identify **the main points shared** across the paragraphs related to "{title}".  
    - **Avoid case-specific details** and focus on general principles.  
    - Present the summary in a **clear and structured manner**.  
    - If there are variations, mention them briefly at the end, without detailing specific cases or articles.  

    **TEXT TO SUMMARIZE:**  
    {extracted_text}
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

def summarize_legal_texts(title, texts):
    """Pipeline completa di summarization per un titolo specifico."""
    return abstractive_summary(title, texts)

# Esempio di utilizzo
summary = summarize_legal_texts("cost", cost)  # 'cost' è il dataset filtrato per il titolo
print(summary)


PROVA 6

In [ ]:
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Carica il modello NLP per segmentare il testo in frasi
nlp = spacy.load("en_core_web_sm")

def split_sentences(texts):
    """Segmenta i paragrafi in frasi utilizzando spaCy."""
    sentences = []
    for text in texts:
        doc = nlp(text)
        sentences.extend([sent.text.strip() for sent in doc.sents])
    return sentences

def optimal_k(embeddings):
    """Determina il numero ottimale di cluster usando il metodo Elbow."""
    distortions = []
    max_clusters = min(10, len(embeddings))  # Non più cluster di quanti siano i dati
    for k in range(2, max_clusters + 1):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)
        distortions.append(kmeans.inertia_)
    return distortions.index(min(distortions)) + 2  # Il punto di minima variazione

def extractive_summary(texts):
    """Seleziona frasi chiave con embeddings e clustering ottimizzato."""
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = split_sentences(texts)
    if len(sentences) <= 5:
        return sentences  # Se il testo è breve, restituisci tutte le frasi

    embeddings = model.encode(sentences)
    k = optimal_k(embeddings)  # Determina il numero ottimale di frasi da estrarre
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)

    # Seleziona frasi più vicine al centroide generale
    centroid = np.mean(embeddings, axis=0)
    similarities = cosine_similarity([centroid], embeddings)[0]
    top_indices = np.argsort(similarities)[-k:]  # Prendi le frasi più simili al centroide
    selected_sentences = [sentences[i] for i in sorted(top_indices)]
    
    return selected_sentences

def abstractive_summary(title, texts):
    """Genera un riassunto astrattivo con Mistral basato sul titolo specifico."""
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    extracted_text = ' '.join(extractive_summary(texts))

    prompt = f"""
    You are a legal text summary model with expertise in analyzing legal documents. Your task is to analyze multiple legal paragraphs related to a given title and generate a concise and structured summary that focuses only on the core legal principles shared across all paragraphs.

    Guidelines:
    
    Avoid mentioning specific articles, case details, judicial chambers, or laws. Focus on the broader concepts that apply universally across different legal contexts.

    Use neutral language and structure the summary in a way that highlights the core principles without being tied to a specific jurisdiction or case.

    If there are variations in how the principles are applied, mention them in very broad terms, without referencing specific norms or cases.

    The summary should describe the principles in a way that makes them understandable in a variety of legal environments.


    **TEXT TO SUMMARIZE:**  
    {extracted_text}
    
    Could you infer a general provision for the set of paragraphs?? 
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

def summarize_legal_texts(title, texts):
    """Pipeline completa di summarization per un titolo specifico."""
    return abstractive_summary(title, texts)

# Esempio di utilizzo
#summary = summarize_legal_texts("cost", cost)  # 'cost' è il dataset filtrato per il titolo
#print(summary)


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    final_summary = summarize_legal_texts(title, texts)
    summary_results[title] = final_summary

In [ ]:
summary_df3 = pd.DataFrame(list(summary_results.items()), columns=['Title', 'Summary'])
for index, row in summary_df3.iterrows():
    print(f"Title: {row['Title']}\nSummary: {row['Summary']}\n{'-'*80}")

PROVA 7

In [ ]:
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Caricamento modello NLP
nlp = spacy.load("en_core_web_sm")

def split_sentences(texts):
    """Segmenta i paragrafi in frasi utilizzando spaCy."""
    sentences = []
    for text in texts:
        doc = nlp(text)
        sentences.extend([sent.text.strip() for sent in doc.sents])
    return sentences

def optimal_k(embeddings):
    """Determina il numero ottimale di cluster usando il Silhouette Score."""
    if len(embeddings) < 3:
        return min(len(embeddings), 5)  # Se pochi dati, scegli un valore basso

    scores = []
    max_clusters = min(10, len(embeddings))

    for k in range(2, max_clusters + 1):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(embeddings)
        score = silhouette_score(embeddings, labels)
        scores.append(score)

    return scores.index(max(scores)) + 2  # Il valore con Silhouette Score più alto

def extractive_summary(texts):
    """Seleziona frasi chiave con embeddings e clustering ottimizzato."""
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = split_sentences(texts)
    if len(sentences) <= 5:
        return sentences  # Se il testo è breve, restituisci tutte le frasi

    embeddings = model.encode(sentences)
    k = optimal_k(embeddings)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)

    # Seleziona frasi più vicine al centroide generale
    centroid = np.mean(embeddings, axis=0)
    similarities = cosine_similarity([centroid], embeddings)[0]
    top_indices = np.argsort(similarities)[-k:]  # Prendi le frasi più simili al centroide
    selected_sentences = [sentences[i] for i in sorted(top_indices)]  # Mantieni l'ordine delle frasi nei testi originali
    
    return selected_sentences

def abstractive_summary(title, texts):
    """Genera un riassunto astrattivo con Mistral basato sul titolo specifico."""
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    extracted_text = ' '.join(extractive_summary(texts))

    prompt = f"""
    You are a legal text summarization model. Your task is to analyze multiple legal paragraphs related to the title "{title}" and generate a **concise and structured summary** that focuses only on the common elements present in all paragraphs.  

    ### **Guidelines:**  
    - Identify **the main points shared** across the paragraphs related to "{title}".  
    - **Avoid case-specific details** and focus on general principles.  
    - Present the summary in a **clear and structured manner**.  
    - If there are variations, mention them briefly at the end, without detailing specific cases or articles.  

    **Example Output:**  
    - These proceedings are considered a step in the action pending before the national court.  
    - The decision on costs is determined by the national court.  
    - Costs incurred in submitting observations to the Court are not recoverable.  

    **TEXT TO SUMMARIZE:**  
    {extracted_text}
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

def summarize_legal_texts(title, texts):
    """Pipeline completa di summarization per un titolo specifico."""
    return abstractive_summary(title, texts)

# Generazione del riassunto per tutti i titoli
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    final_summary = summarize_legal_texts(title, texts)
    summary_results[title] = final_summary

# Output dei risultati
for title, summary in summary_results.items():
    print(f"Title: {title}\nSummary:\n{summary}\n{'-'*50}")


PROVA 8

In [ ]:
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Caricamento modello NLP
nlp = spacy.load("en_core_web_sm")

def split_sentences(texts):
    """Segmenta i paragrafi in frasi utilizzando spaCy."""
    sentences = []
    for text in texts:
        doc = nlp(text)
        sentences.extend([sent.text.strip() for sent in doc.sents])
    return sentences

def filter_similar_sentences(sentences, embeddings, threshold=0.8):
    """Rimuove frasi troppo simili tra loro per evitare ripetizioni."""
    filtered = []
    seen = set()
    for i, sent in enumerate(sentences):
        sim_scores = cosine_similarity([embeddings[i]], embeddings)[0]
        if not any(sim > threshold for j, sim in enumerate(sim_scores) if j in seen):
            filtered.append(sent)
            seen.add(i)
    return filtered

def optimal_k(embeddings):
    """Determina il numero ottimale di cluster usando il Silhouette Score."""
    if len(embeddings) < 3:
        return min(len(embeddings), 5)  # Se pochi dati, scegli un valore basso

    scores = []
    max_clusters = min(8, len(embeddings))  # Ridotto per stabilità

    for k in range(2, max_clusters + 1):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(embeddings)
        score = silhouette_score(embeddings, labels)
        scores.append(score)

    return scores.index(max(scores)) + 2

def extractive_summary(texts):
    """Seleziona frasi chiave con embeddings e clustering ottimizzato."""
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = split_sentences(texts)
    if len(sentences) <= 5:
        return sentences

    embeddings = model.encode(sentences)
    k = optimal_k(embeddings)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)

    # Seleziona frasi più vicine al centroide generale
    centroid = np.mean(embeddings, axis=0)
    similarities = cosine_similarity([centroid], embeddings)[0]
    top_indices = np.argsort(similarities)[-k:]  # Frasi più vicine al centroide
    selected_sentences = [sentences[i] for i in sorted(top_indices)]
    
    return filter_similar_sentences(selected_sentences, embeddings)  # Rimuove frasi ridondanti

def abstractive_summary(title, texts):
    """Genera un riassunto astrattivo con Mistral basato sul titolo specifico."""
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    extracted_text = ' '.join(extractive_summary(texts))

    prompt = f"""
    You are a legal summarization model. Analyze the legal paragraphs related to "{title}" and generate a **concise and structured summary** by focusing on key common elements.

    ### **Guidelines:**
    - Extract **the main points shared** across all paragraphs.
    - **Do NOT include details specific to individual cases.**
    - Present the summary as a **clear and structured list**.
    - If variations exist, mention them at the end, without excessive detail.

    **Example Output:**
    - These proceedings are part of an action before the national court.
    - The national court decides on costs.
    - Costs for submitting observations to the Court are not recoverable.

    **TEXT TO SUMMARIZE:**
    {extracted_text}
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

def summarize_legal_texts(title, texts):
    """Pipeline completa di summarization per un titolo specifico."""
    return abstractive_summary(title, texts)

# Generazione del riassunto per tutti i titoli
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    final_summary = summarize_legal_texts(title, texts)
    summary_results[title] = final_summary

# Output dei risultati
for title, summary in summary_results.items():
    print(f"Title: {title}\nSummary:\n{summary}\n{'-'*50}")


In [ ]:
pip install kneed

PROVA 9

In [ ]:
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from kneed import KneeLocator

# Carica il modello NLP per segmentare il testo in frasi
nlp = spacy.load("en_core_web_sm")

def split_sentences(texts):
    """Segmenta i paragrafi in frasi utilizzando spaCy."""
    sentences = []
    for text in texts:
        doc = nlp(text)
        sentences.extend([sent.text.strip() for sent in doc.sents])
    return sentences

def optimal_k(embeddings):
    """Determina il numero ottimale di cluster con il metodo Elbow usando KneeLocator."""
    distortions = []
    max_clusters = min(10, len(embeddings))
    k_range = range(2, max_clusters + 1)

    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)
        distortions.append(kmeans.inertia_)

    kneedle = KneeLocator(k_range, distortions, curve="convex", direction="decreasing")
    return kneedle.elbow if kneedle.elbow else 2  # Default a 2 se non trova il ginocchio

def extractive_summary(texts):
    """Seleziona frasi chiave con embeddings e clustering ottimizzato."""
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = split_sentences(texts)
    if len(sentences) <= 5:
        return sentences  

    embeddings = model.encode(sentences)
    k = optimal_k(embeddings)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)

    # Trova la frase più rappresentativa per ciascun cluster
    cluster_centers = kmeans.cluster_centers_
    selected_sentences = []

    for i in range(k):
        cluster_indices = np.where(kmeans.labels_ == i)[0]
        cluster_embeddings = embeddings[cluster_indices]
        similarities = cosine_similarity([cluster_centers[i]], cluster_embeddings)[0]
        best_idx = cluster_indices[np.argmax(similarities)]  # Frase più vicina al centro del cluster
        selected_sentences.append(sentences[best_idx])

    return sorted(selected_sentences, key=sentences.index)  # Mantiene ordine originale

def abstractive_summary(title, texts):
    """Genera un riassunto astrattivo con Mistral basato sul titolo specifico."""
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    extracted_text = ' '.join(extractive_summary(texts))

    prompt = f"""
    You are an expert at summarizing legal documents. Your task is to generate a concise summary of the key legal principles in the paragraphs provided.

    - DO NOT mention specific cases, articles, or jurisdictions.
    - Focus ONLY on general legal concepts that apply widely.
    - Keep the summary under 100 words.

    **TEXT TO SUMMARIZE:**
    {extracted_text[:2000]} # Truncate text to avoid excess

    Provide a generalized summary. 
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

def summarize_legal_texts(title, texts):
    """Pipeline completa di summarization per un titolo specifico."""
    return abstractive_summary(title, texts)

# Esempio di utilizzo
#summary = summarize_legal_texts("cost", cost)  # 'cost' è il dataset filtrato per il titolo
#print(summary)


In [ ]:
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    final_summary = summarize_legal_texts(title, texts)
    summary_results[title] = final_summary

In [ ]:
# Output dei risultati
for title, summary in summary_results.items():
    print(f"Title: {title}\nSummary:\n{summary}\n{'-'*50}")

PROVA 9 CON VALUTAZIONE

In [ ]:
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from kneed import KneeLocator
from rouge_score import rouge_scorer

# Carica il modello NLP per segmentare il testo in frasi
nlp = spacy.load("en_core_web_sm")

def split_sentences(texts):
    """Segmenta i paragrafi in frasi utilizzando spaCy."""
    sentences = []
    for text in texts:
        doc = nlp(text)
        sentences.extend([sent.text.strip() for sent in doc.sents])
    return sentences

def optimal_k(embeddings):
    """Determina il numero ottimale di cluster con il metodo Elbow usando KneeLocator."""
    distortions = []
    max_clusters = min(10, len(embeddings))
    k_range = range(2, max_clusters + 1)

    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)
        distortions.append(kmeans.inertia_)

    kneedle = KneeLocator(k_range, distortions, curve="convex", direction="decreasing")
    return kneedle.elbow if kneedle.elbow else 2  # Default a 2 se non trova il ginocchio

def extractive_summary(texts):
    """Seleziona frasi chiave con embeddings e clustering ottimizzato."""
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = split_sentences(texts)
    if len(sentences) <= 5:
        return sentences  

    embeddings = model.encode(sentences)
    k = optimal_k(embeddings)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)

    # Trova la frase più rappresentativa per ciascun cluster
    cluster_centers = kmeans.cluster_centers_
    selected_sentences = []

    for i in range(k):
        cluster_indices = np.where(kmeans.labels_ == i)[0]
        cluster_embeddings = embeddings[cluster_indices]
        similarities = cosine_similarity([cluster_centers[i]], cluster_embeddings)[0]
        best_idx = cluster_indices[np.argmax(similarities)]  # Frase più vicina al centro del cluster
        selected_sentences.append(sentences[best_idx])

    return sorted(selected_sentences, key=sentences.index)  # Mantiene ordine originale

def abstractive_summary(title, texts):
    """Genera un riassunto astrattivo con Mistral basato sul titolo specifico."""
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    extracted_text = ' '.join(extractive_summary(texts))

    prompt = f"""
    You are an expert at summarizing legal documents. Your task is to generate a concise summary of the key legal principles in the paragraphs provided.

    - DO NOT mention specific cases, articles, or jurisdictions.
    - Focus ONLY on general legal concepts that apply widely.
    - Keep the summary under 100 words.

    **TEXT TO SUMMARIZE:**
    {extracted_text[:2000]} # Truncate text to avoid excess

    Provide a generalized summary. 
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

def evaluate_summary(original_text, summary):
    """Calcola metriche di valutazione del riassunto."""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(original_text, summary)

    # Calcola la similarità semantica con Sentence-BERT
    model = SentenceTransformer('all-MiniLM-L6-v2')
    emb_original = model.encode([original_text])
    emb_summary = model.encode([summary])
    semantic_similarity = cosine_similarity(emb_original, emb_summary)[0][0]

    # Calcola la compressione del testo
    compression_ratio = len(summary) / len(original_text) if len(original_text) > 0 else 0

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "Semantic Similarity": semantic_similarity,
        "Compression Ratio": compression_ratio
    }

def summarize_legal_texts(title, texts):
    """Pipeline completa di summarization per un titolo specifico e valutazione."""
    summary = abstractive_summary(title, texts)
    original_text = " ".join(texts)
    evaluation_metrics = evaluate_summary(original_text, summary)
    return summary, evaluation_metrics

# 🚀 Esegui la summarization su tutti i titoli del dataset
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    final_summary, metrics = summarize_legal_texts(title, texts)
    
    summary_results[title] = {
        "summary": final_summary,
        "metrics": metrics
    }

# Stampa i risultati
for title, result in summary_results.items():
    print(f"Title: {title}")
    print(f"Summary: {result['summary']}")
    print(f"Metrics: {result['metrics']}")
    print("-" * 50)


PROVA 10 CON VALUTAZIONE

In [ ]:
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from kneed import KneeLocator
from rouge_score import rouge_scorer

# Carica modello NLP per lo split in frasi
nlp = spacy.load("en_core_web_sm")

def split_sentences(texts):
    sentences = []
    for text in texts:
        doc = nlp(text)
        sentences.extend([sent.text.strip() for sent in doc.sents])
    return sentences

def optimal_k(embeddings):
    distortions = []
    max_clusters = min(10, len(embeddings))
    k_range = range(2, max_clusters + 1)

    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)
        distortions.append(kmeans.inertia_)

    kneedle = KneeLocator(k_range, distortions, curve="convex", direction="decreasing")
    return kneedle.elbow if kneedle.elbow else 2

def filter_redundant(sentences, embeddings, threshold=0.8):
    filtered = []
    used = []
    for i, emb in enumerate(embeddings):
        if any(cosine_similarity([emb], [embeddings[j]])[0][0] > threshold for j in used):
            continue
        filtered.append(sentences[i])
        used.append(i)
    return filtered

def extractive_summary(texts):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = split_sentences(texts)
    if len(sentences) <= 5:
        return sentences  

    embeddings = model.encode(sentences)
    k = optimal_k(embeddings)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)

    cluster_centers = kmeans.cluster_centers_
    selected_sentences = []

    for i in range(k):
        cluster_indices = np.where(kmeans.labels_ == i)[0]
        cluster_embeddings = embeddings[cluster_indices]
        similarities = cosine_similarity([cluster_centers[i]], cluster_embeddings)[0]

        # Pondera la posizione nel testo (più vicina all'inizio = meglio)
        scores = [
            sim * (1 + 0.2 * (1.0 - cluster_indices[j] / len(sentences)))
            for j, sim in enumerate(similarities)
        ]
        best_idx = cluster_indices[np.argmax(scores)]
        selected_sentences.append(sentences[best_idx])

    selected_embeddings = model.encode(selected_sentences)
    filtered_sentences = filter_redundant(selected_sentences, selected_embeddings)

    return sorted(filtered_sentences, key=sentences.index)

def abstractive_summary(title, texts):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    extracted_text = ' '.join(extractive_summary(texts))

    prompt = f"""
    You are a legal expert AI. You are given a set of representative sentences extracted from a legal document on the topic: "{title}".

    Your task is to produce a clear, generalized summary of the core legal principles, avoiding reference to specific laws, articles, or jurisdictions.

    - Use simple and formal language.
    - The summary must be under 100 words.
    - Rephrase and restructure to ensure natural flow and coherence.

    Sentences to summarize:
    {extracted_text[:2000]}
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

def evaluate_summary(original_text, summary):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(original_text, summary)

    model = SentenceTransformer('all-MiniLM-L6-v2')
    emb_original = model.encode([original_text])
    emb_summary = model.encode([summary])
    semantic_similarity = cosine_similarity(emb_original, emb_summary)[0][0]

    compression_ratio = len(summary) / len(original_text) if len(original_text) > 0 else 0

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "Semantic Similarity": semantic_similarity,
        "Compression Ratio": compression_ratio
    }

def summarize_legal_texts(title, texts):
    summary = abstractive_summary(title, texts)
    original_text = " ".join(texts)
    evaluation_metrics = evaluate_summary(original_text, summary)
    return summary, evaluation_metrics

# 🚀 Applica la pipeline a tutti i titoli
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    final_summary, metrics = summarize_legal_texts(title, texts)
    
    summary_results[title] = {
        "summary": final_summary,
        "metrics": metrics
    }

# 📄 Output finale
for title, result in summary_results.items():
    print(f"Title: {title}")
    print(f"Summary: {result['summary']}")
    print(f"Metrics: {result['metrics']}")
    print("-" * 50)


PROVA 11

In [ ]:
import re
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from kneed import KneeLocator
from rouge_score import rouge_scorer
from joblib import Memory

# Configurazione cache per embeddings (migliora le prestazioni)
memory = Memory("cache_dir", verbose=0)

# Caricamento modelli
nlp = spacy.load("en_core_web_sm")  # Per lo split in frasi
legal_embedding_model = SentenceTransformer('nlpaueb/legal-bert-small-uncased')  # Modello legal-specific

# ====================== PREPROCESSING ======================
def preprocess_legal_text(text):
    """
    Normalizza il testo rimuovendo dettagli specifici del dominio legale.
    Esempio: "Case C-123/45" -> "[CASE_REF]", "the claimant" -> "[PARTY]"
    """
    text = re.sub(r'(Case\s[A-Za-z0-9-/]+)|(Article\s\d+)', '[LEGAL_REF]', text)
    text = re.sub(r'(the\s(defendant|claimant|appellant))', '[PARTY]', text, flags=re.IGNORECASE)
    return text

# ====================== FASE ESTRATTIVA ======================
@memory.cache
def encode_sentences(sentences):
    """Calcola embeddings con caching per evitare ricalcoli"""
    return legal_embedding_model.encode(sentences)

def split_sentences(texts):
    """Divide i testi in frasi usando Spacy, con preprocessing"""
    sentences = []
    for text in texts:
        doc = nlp(preprocess_legal_text(text))
        sentences.extend([sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10])
    return sentences

def optimal_k(embeddings):
    """Trova il numero ottimale di cluster usando il metodo del gomito"""
    distortions = []
    max_clusters = min(10, len(embeddings))
    for k in range(2, max_clusters + 1):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)
        distortions.append(kmeans.inertia_)
    return KneeLocator(range(2, max_clusters + 1), distortions, curve="convex", direction="decreasing").elbow or 2

def extractive_summary(texts):
    """Seleziona le frasi più rappresentative da un insieme di testi"""
    sentences = split_sentences(texts)
    if len(sentences) <= 3:
        return sentences

    embeddings = encode_sentences(sentences)
    k = optimal_k(embeddings)
    
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)
    
    selected_sentences = []
    for cluster_id in range(k):
        cluster_mask = (kmeans.labels_ == cluster_id)
        cluster_embeddings = embeddings[cluster_mask]
        centroid = kmeans.cluster_centers_[cluster_id]
        
        # Seleziona la frase più vicina al centroide
        closest_idx = np.argmax(cosine_similarity([centroid], cluster_embeddings)[0])
        selected_sentences.append(sentences[np.where(cluster_mask)[0][closest_idx]])
    
    return selected_sentences

# ====================== FASE ASTRATTIVA ======================
def generate_legal_prompt(title, extracted_sentences):
    """Crea un prompt strutturato per il modello LLM"""
    return f"""
You are a senior legal expert summarizing principles from court documents about: {title}.

**Guidelines**:
1. Use ONLY generic legal terms (e.g., "[PARTY]", "[LEGAL_REF]").
2. Describe COMMON PATTERNS (e.g., "Courts typically...").
3. NEVER mention specific cases, articles, or names.
4. Keep summary under 80 words.

**Example Output**:
"For [PARTY] to establish liability, it must generally prove [LEGAL_REF] was violated."

**Text to Summarize**:
{" ".join(extracted_sentences)[:1500]}
"""

def abstractive_summary(title, extracted_sentences):
    """Genera un riassunto astratto generalizzato"""
    if not extracted_sentences:
        return "No summary available."

    prompt = generate_legal_prompt(title, extracted_sentences)
    response = ollama.chat(
        model='mistral',
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.3}  # Riduci creatività per output più conservativo
    )
    return response['message']['content']

# ====================== VALUTAZIONE ======================
def evaluate_summary(original_text, summary):
    """Calcola metriche di qualità con focus sulla generalizzazione"""
    # Metriche standard
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(original_text, summary)
    
    # Metrica custom per generalizzazione
    legal_refs = len(re.findall(r'\[LEGAL_REF\]|\[PARTY\]', summary))
    specificity_penalty = 1.0 if legal_refs >= 2 else 0.5  # Premia l'uso di placeholder
    
    return {
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "Generalization_Score": specificity_penalty,
        "Length_Ratio": len(summary) / len(original_text) if original_text else 0
    }

# ====================== PIPELINE COMPLETA ======================
def summarize_legal_section(title, texts):
    """
    Pipeline end-to-end:
    1. Preprocessing -> 2. Estrazione -> 3. Astrazione -> 4. Valutazione
    """
    if not texts:
        return None, {"error": "No input texts"}
    
    # Fase estrattiva
    extracted = extractive_summary(texts)
    
    # Fase astrattiva
    summary = abstractive_summary(title, extracted)
    
    # Valutazione
    metrics = evaluate_summary(" ".join(texts), summary)
    
    return summary, metrics



texts = get_texts_for_title(event_log, 'cost')
summary, metrics = summarize_legal_section('cost', texts)



print(f"Summary: {summary}")
print(f"Metrics: {metrics}\n")




PROVA 12

In [ ]:
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from kneed import KneeLocator
from rouge_score import rouge_scorer

# Carica modello NLP per lo split in frasi
nlp = spacy.load("en_core_web_sm")

def split_sentences(texts):
    sentences = []
    for text in texts:
        doc = nlp(text)
        sentences.extend([sent.text.strip() for sent in doc.sents])
    return sentences

def optimal_k(embeddings):
    distortions = []
    max_clusters = min(10, len(embeddings))
    k_range = range(2, max_clusters + 1)

    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)
        distortions.append(kmeans.inertia_)

    kneedle = KneeLocator(k_range, distortions, curve="convex", direction="decreasing")
    return kneedle.elbow if kneedle.elbow else 2

def filter_redundant(sentences, embeddings, threshold=0.8):
    filtered = []
    used = []
    for i, emb in enumerate(embeddings):
        if any(cosine_similarity([emb], [embeddings[j]])[0][0] > threshold for j in used):
            continue
        filtered.append(sentences[i])
        used.append(i)
    return filtered

def extractive_summary(texts):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = split_sentences(texts)
    if len(sentences) <= 5:
        return sentences  

    embeddings = model.encode(sentences)
    k = optimal_k(embeddings)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)

    cluster_centers = kmeans.cluster_centers_
    selected_sentences = []

    for i in range(k):
        cluster_indices = np.where(kmeans.labels_ == i)[0]
        cluster_embeddings = embeddings[cluster_indices]
        similarities = cosine_similarity([cluster_centers[i]], cluster_embeddings)[0]

        # Pondera la posizione nel testo (più vicina all'inizio = meglio)
        scores = [
            sim * (1 + 0.2 * (1.0 - cluster_indices[j] / len(sentences)))
            for j, sim in enumerate(similarities)
        ]
        best_idx = cluster_indices[np.argmax(scores)]
        selected_sentences.append(sentences[best_idx])

    selected_embeddings = model.encode(selected_sentences)
    filtered_sentences = filter_redundant(selected_sentences, selected_embeddings)

    return sorted(filtered_sentences, key=sentences.index)

def abstractive_summary(title, texts):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    extracted_text = ' '.join(extractive_summary(texts))

    prompt = f"""
    You are a legal expert AI. Your task is to write a concise and generalized summary of the key legal principles discussed in the following content, which is under the topic: "{title}".

    Please follow these instructions:
    - DO NOT reference specific laws, articles, jurisdictions, case numbers, court decisions, dates, or institutions.
    - Focus only on the underlying **legal logic**, **principles**, and **general interpretations**.
    - Use clear, formal, and neutral language.
    - Ensure natural flow and coherence, avoiding list formats.
    - The summary must be a single paragraph under 100 words.

    Here are the key extracted sentences:
    {extracted_text[:2000]}
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

def evaluate_summary(original_text, summary):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(original_text, summary)

    model = SentenceTransformer('all-MiniLM-L6-v2')
    emb_original = model.encode([original_text])
    emb_summary = model.encode([summary])
    semantic_similarity = cosine_similarity(emb_original, emb_summary)[0][0]

    compression_ratio = len(summary) / len(original_text) if len(original_text) > 0 else 0

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "Semantic Similarity": semantic_similarity,
        "Compression Ratio": compression_ratio
    }

def summarize_legal_texts(title, texts):
    summary = abstractive_summary(title, texts)
    original_text = " ".join(texts)
    evaluation_metrics = evaluate_summary(original_text, summary)
    return summary, evaluation_metrics

# 🚀 Applica la pipeline a tutti i titoli
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    final_summary, metrics = summarize_legal_texts(title, texts)
    
    summary_results[title] = {
        "summary": final_summary,
        "metrics": metrics
    }

# 📄 Output finale
for title, result in summary_results.items():
    print(f"Title: {title}")
    print(f"Summary: {result['summary']}")
    print(f"Metrics: {result['metrics']}")
    print("-" * 50)


PROVA 13

In [ ]:
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from kneed import KneeLocator
from rouge_score import rouge_scorer
from bert_score import score

# Carica modello NLP per lo split in frasi
nlp = spacy.load("en_core_web_sm")

def split_sentences(texts):
    sentences = []
    for text in texts:
        doc = nlp(text)
        sentences.extend([sent.text.strip() for sent in doc.sents])
    return sentences

def optimal_k(embeddings):
    distortions = []
    max_clusters = min(10, len(embeddings))
    k_range = range(2, max_clusters + 1)

    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)
        distortions.append(kmeans.inertia_)

    kneedle = KneeLocator(k_range, distortions, curve="convex", direction="decreasing")
    return kneedle.elbow if kneedle.elbow else 2

def filter_redundant(sentences, embeddings, threshold=0.8):
    filtered = []
    used = []
    for i, emb in enumerate(embeddings):
        if any(cosine_similarity([emb], [embeddings[j]])[0][0] > threshold for j in used):
            continue
        filtered.append(sentences[i])
        used.append(i)
    return filtered

def extractive_summary(texts):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = split_sentences(texts)
    if len(sentences) <= 5:
        return sentences  

    embeddings = model.encode(sentences)
    k = optimal_k(embeddings)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)

    cluster_centers = kmeans.cluster_centers_
    selected_sentences = []

    for i in range(k):
        cluster_indices = np.where(kmeans.labels_ == i)[0]
        cluster_embeddings = embeddings[cluster_indices]
        similarities = cosine_similarity([cluster_centers[i]], cluster_embeddings)[0]

        # Pondera la posizione nel testo (più vicina all'inizio = meglio)
        scores = [
            sim * (1 + 0.2 * (1.0 - cluster_indices[j] / len(sentences)))
            for j, sim in enumerate(similarities)
        ]
        best_idx = cluster_indices[np.argmax(scores)]
        selected_sentences.append(sentences[best_idx])

    selected_embeddings = model.encode(selected_sentences)
    filtered_sentences = filter_redundant(selected_sentences, selected_embeddings)

    return sorted(filtered_sentences, key=sentences.index)

def abstractive_summary(title, texts):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    extracted_text = ' '.join(extractive_summary(texts))

    prompt = f"""
    You are a legal expert AI. Your task is to write a concise and generalized summary of the key legal principles discussed in the following content, which is under the topic: "{title}".

    Please follow these instructions:
    - DO NOT reference specific laws, articles, jurisdictions, case numbers, court decisions, dates, or institutions.
    - Focus only on the underlying **legal logic**, **principles**, and **general interpretations**.
    - Use clear, formal, and neutral language.
    - Ensure natural flow and coherence, avoiding list formats.
    - The summary must be a single paragraph under 100 words.

    Here are the key extracted sentences:
    {extracted_text[:2000]}
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

def evaluate_summary(original_text, summary):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(original_text, summary)

    model = SentenceTransformer('all-MiniLM-L6-v2')
    emb_original = model.encode([original_text])
    emb_summary = model.encode([summary])
    semantic_similarity = cosine_similarity(emb_original, emb_summary)[0][0]

    compression_ratio = len(summary) / len(original_text) if len(original_text) > 0 else 0

    P, R, F1 = score([summary], [original_text], lang="en", verbose=False)
    bertscore_precision = P.mean().item()
    bertscore_recall = R.mean().item()
    bertscore_f1 = F1.mean().item()

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "Semantic Similarity": semantic_similarity,
        "Compression Ratio": compression_ratio,
        "BERTScore Precision": bertscore_precision,
        "BERTScore Recall": bertscore_recall,
        "BERTScore F1": bertscore_f1
    }

def summarize_legal_texts(title, texts):
    summary = abstractive_summary(title, texts)
    original_text = " ".join(texts)
    evaluation_metrics = evaluate_summary(original_text, summary)
    return summary, evaluation_metrics

# 🚀 Applica la pipeline a tutti i titoli
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    final_summary, metrics = summarize_legal_texts(title, texts)
    
    summary_results[title] = {
        "summary": final_summary,
        "metrics": metrics
    }

# 📄 Output finale
for title, result in summary_results.items():
    print(f"Title: {title}")
    print(f"Summary: {result['summary']}")
    print(f"Metrics: {result['metrics']}")
    print("-" * 50)


PROVA 14

In [ ]:
import ollama
import numpy as np
import spacy
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from kneed import KneeLocator
from rouge_score import rouge_scorer
from bert_score import score

# Carica modello NLP per lo split in frasi
nlp = spacy.load("en_core_web_sm")

def split_sentences(texts):
    sentences = []
    for text in texts:
        doc = nlp(text)
        sentences.extend([sent.text.strip() for sent in doc.sents])
    return sentences

def optimal_k(embeddings):
    distortions = []
    max_clusters = min(10, len(embeddings))
    k_range = range(2, max_clusters + 1)

    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(embeddings)
        distortions.append(kmeans.inertia_)

    kneedle = KneeLocator(k_range, distortions, curve="convex", direction="decreasing")
    return kneedle.elbow if kneedle.elbow else 2

def filter_redundant(sentences, embeddings, threshold=0.8):
    filtered = []
    used = []
    for i, emb in enumerate(embeddings):
        if any(cosine_similarity([emb], [embeddings[j]])[0][0] > threshold for j in used):
            continue
        filtered.append(sentences[i])
        used.append(i)
    return filtered

def extractive_summary(texts):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = split_sentences(texts)
    if len(sentences) <= 5:
        return sentences  

    embeddings = model.encode(sentences)
    k = optimal_k(embeddings)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)

    cluster_centers = kmeans.cluster_centers_
    selected_sentences = []

    for i in range(k):
        cluster_indices = np.where(kmeans.labels_ == i)[0]
        cluster_embeddings = embeddings[cluster_indices]
        similarities = cosine_similarity([cluster_centers[i]], cluster_embeddings)[0]

        # Pondera la posizione nel testo (più vicina all'inizio = meglio)
        scores = [
            sim * (1 + 0.2 * (1.0 - cluster_indices[j] / len(sentences)))
            for j, sim in enumerate(similarities)
        ]
        best_idx = cluster_indices[np.argmax(scores)]
        selected_sentences.append(sentences[best_idx])

    selected_embeddings = model.encode(selected_sentences)
    filtered_sentences = filter_redundant(selected_sentences, selected_embeddings)

    return sorted(filtered_sentences, key=sentences.index)

def abstractive_summary(title, texts):
    if not texts:
        return "Nessun testo disponibile per il riassunto."

    extracted_text = ' '.join(extractive_summary(texts))

    prompt = f"""
    You are a legal expert. Your task is to write a concise and generalized summary of the key legal principles covered in the content below, which relates to the topic: "{title}".

    To do this, you must:
    - Identify and summarize the general legal principles and underlying logic, without going into the specific details of laws, articles, jurisdictions or legal decisions.
    - Use clear, formal and neutral language, without resorting to lists or broken formats.
    - Ensure a natural flow and coherence, keeping the summary under 100 characters.
    
    Here are the key extracted sentences:
    {extracted_text[:2000]}
    """

    response = ollama.chat(model='mistral', messages=[{"role": "user", "content": prompt}])
    return response['message']['content']

def evaluate_summary(original_text, summary):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(original_text, summary)

    model = SentenceTransformer('all-MiniLM-L6-v2')
    emb_original = model.encode([original_text])
    emb_summary = model.encode([summary])
    semantic_similarity = cosine_similarity(emb_original, emb_summary)[0][0]

    compression_ratio = len(summary) / len(original_text) if len(original_text) > 0 else 0

    P, R, F1 = score([summary], [original_text], lang="en", verbose=False)
    bertscore_precision = P.mean().item()
    bertscore_recall = R.mean().item()
    bertscore_f1 = F1.mean().item()

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "Semantic Similarity": semantic_similarity,
        "Compression Ratio": compression_ratio,
        "BERTScore Precision": bertscore_precision,
        "BERTScore Recall": bertscore_recall,
        "BERTScore F1": bertscore_f1
    }

def summarize_legal_texts(title, texts):
    summary = abstractive_summary(title, texts)
    original_text = " ".join(texts)
    evaluation_metrics = evaluate_summary(original_text, summary)
    return summary, evaluation_metrics

# 🚀 Applica la pipeline a tutti i titoli
summary_results = {}

for title in [t.label for t in new_net.transitions if t.label is not None]:
    texts = get_texts_for_title(event_log, title)
    final_summary, metrics = summarize_legal_texts(title, texts)
    
    summary_results[title] = {
        "summary": final_summary,
        "metrics": metrics
    }

# 📄 Output finale
for title, result in summary_results.items():
    print(f"Title: {title}")
    print(f"Summary: {result['summary']}")
    print(f"Metrics: {result['metrics']}")
    print("-" * 50)


In [ ]:
from typing import List, Dict
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')

def split_into_chunks(text: str, max_chunk_size: int = 500) -> List[str]:
    """Divide il testo in chunk di dimensioni gestibili."""
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chunk_size:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

In [ ]:
def abstractive_summarization(texts: List[str], prompt_template: str, title: str) -> str:
    """Combina i chunk e genera un riassunto astrattivo con Mistral."""
    combined_text = " ".join(texts)
    prompt = prompt_template.format(title=title, text=combined_text)  # Aggiungi title qui
    
    response = ollama.generate(
        model="mistral",
        prompt=prompt,  # Corretto typo ("prompt" non "prompt")
        options={"temperature": 0.3}
    )
    return response["response"]

def legal_summarization_mapreduce(texts: List[str], title: str) -> str:
    """Pipeline completa per summarization legale."""
    # 1. Suddividi ogni testo in chunk
    chunks = []
    for text in texts:
        chunks.extend(split_into_chunks(text))
    
    # 2. Riassunto estrattivo per ogni chunk (Map)
    extractive_summaries = [extractive_summarization(chunk) for chunk in chunks]
    
    # 3. Riassunto astrattivo globale (Reduce)
    prompt_template = """
    Sei un assistente legale. Genera un riassunto conciso del seguente testo, 
    evidenziando le informazioni più rilevanti per il titolo: "{title}".
    Se ci sono contraddizioni, menzionale entrambe le prospettive.
    
    Testo: {text}
    """
    final_summary = abstractive_summarization(extractive_summaries, prompt_template, title)  # Passa title
    return final_summary

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def extractive_summarization(text: str, top_n: int = 3) -> str:
    """Seleziona le frasi più importanti usando TF-IDF."""
    sentences = sent_tokenize(text)
    if len(sentences) <= top_n:
        return text
    
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(sentences)
    sentence_scores = np.sum(tfidf_matrix.toarray(), axis=1)
    top_sentences = [sentences[i] for i in np.argsort(sentence_scores)[-top_n:]]
    return " ".join(top_sentences)

In [ ]:
texts = [
    "Il tribunale ha stabilito che la difesa è vincitrice, in accordo con l'art. 123.",
    "In appello, la difesa è stata invece condannata ai sensi dell'art. 456."
]
title = "Esito della causa"

summary = legal_summarization_mapreduce(texts, title)
print(summary)

In [ ]:
event_log

In [ ]:
# 10. Visualizza la distribuzione di una colonna categorica
print("\nDistribuzione dei valori in 'concept:name':")
print(event_log['concept:name'].value_counts().head(23))  

In [ ]:
titles = [t.label for t in new_net.transitions if t.label is not None]
def get_texts_for_title(event_log, title):
    return event_log[event_log['concept:name'] == title]['concept:name:section'].tolist()  

In [ ]:
from transformers import pipeline, AutoTokenizer

# Inizializza tokenizer e summarizer
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
summarizer = pipeline("summarization", model=model_name)

texts = get_texts_for_title(event_log, "cost")

# Funzione helper per troncare in base ai token
def truncate_text(text, max_tokens=1024):
    tokens = tokenizer.encode(text, truncation=True, max_length=max_tokens)
    return tokenizer.decode(tokens, skip_special_tokens=True)

# 3. Applica la fase "Map"
mapped_summaries = []
for text in texts:  # limitiamo per test
    shortened = truncate_text(text, max_tokens=1024)
    summary = summarizer(shortened, max_length=100, min_length=30, do_sample=False)[0]['summary_text']
    mapped_summaries.append(summary)

# 4. Fase "Reduce"
joined_summary = " ".join(mapped_summaries)
joined_shortened = truncate_text(joined_summary, max_tokens=1024)
final_summary = summarizer(joined_shortened, max_length=150, min_length=50, do_sample=False)[0]['summary_text']

# 5. Risultato
print("🔹 Summary finale per il titolo 'cost':\n")
print(final_summary)


RAG

In [ ]:
pip install faiss-cpu sentence-transformers pandas

In [ ]:
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
import subprocess

# Carica embedding model (puoi usare anche modelli legali specifici)
embedder = SentenceTransformer('all-MiniLM-L6-v2')


def get_texts_for_title(event_log, title):
    return event_log[event_log['concept:name'] == title]['concept:name:section'].tolist()

def build_faiss_index(texts):
    embeddings = embedder.encode(texts, convert_to_tensor=False)
    index = faiss.IndexFlatL2(len(embeddings[0]))
    index.add(np.array(embeddings).astype('float32'))
    return index, embeddings

def retrieve_top_k(texts, index, query, k=5):
    query_vec = embedder.encode([query])[0].astype('float32')
    D, I = index.search(np.array([query_vec]), k)
    return [texts[i] for i in I[0]]

def generate_summary_with_ollama(title, retrieved_texts):
    joined = "\n\n".join([f"{i+1}. {txt}" for i, txt in enumerate(retrieved_texts)])
    prompt = (
        f"""
        You are a legal assistant specialized in analyzing legal texts. 
        Your goal is to explain the general legal concept that emerges when analyzing the following legal texts grouped under the title: {title}. You are not summarizing each paragraph individually — instead, abstract the underlying legal theme that unites them all, even if they differ in specific facts or rulings.

        Avoid references to specific directives, articles, or case numbers. Focus only on what the texts collectively reveal about the legal principle, doctrine, or procedural rule they reflect.

        ### Input:
        The legal paragraphs related to {title} are as follows:\n\n{joined}


        ### Output format:
        “The concept that unites these paragraphs is as follows:

        [Provide a concise, structured explanation of the common legal theme.]

        However, there are variations in [briefly mention any recurring exceptions or deviations, without going into detail].”

        """
    )
    
    # Chiamata al modello Mistral via Ollama
    result = subprocess.run(
        ["ollama", "run", "mistral"],
        input=prompt.encode("utf-8"),
        capture_output=True
    )
    return result.stdout.decode("utf-8").strip()

# Main loop
titles = [t.label for t in new_net.transitions if t.label is not None]
summaries = {}

for title in titles:
    texts = get_texts_for_title(event_log, title)
    
    if len(texts) < 3:
        print(f"⚠️ Titolo '{title}' ha pochi testi, salto...")
        continue

    print(f"📚 Indicizzazione: {title} ({len(texts)} testi)")
    index, _ = build_faiss_index(texts)

    retrieved = retrieve_top_k(texts, index, title, k=5)
    summary = generate_summary_with_ollama(title, retrieved)
    summaries[title] = summary

    print(f"✅ {title}:\n{summary}\n{'-'*50}")


In [ ]:
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
import subprocess
import traceback

# ---- 1. Setup del modello di embedding ----
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# ---- 2. Funzione: recupera i testi associati a un titolo ----
def get_texts_for_title(event_log, title):
    return event_log[event_log['concept:name'] == title]['concept:name:section'].tolist()

# ---- 3. Funzione: costruisci FAISS index ----
def build_faiss_index(texts):
    embeddings = embedder.encode(texts, convert_to_tensor=False)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings).astype('float32'))
    return index, embeddings

# ---- 4. Funzione: recupera i testi più rilevanti ----
def retrieve_top_k(texts, index, query, k=5, similarity_threshold=0.3):
    query_vec = embedder.encode([query])[0].astype('float32')
    D, I = index.search(np.array([query_vec]), k)
    
    retrieved = []
    for distance, idx in zip(D[0], I[0]):
        if idx != -1:
            similarity = 1 / (1 + distance)  # Conversione distanza → similarità (approssimato)
            if similarity > similarity_threshold:
                retrieved.append(texts[idx])
    
    return retrieved

# ---- 5. Funzione: genera il prompt migliorato ----
def build_prompt(title, retrieved_texts):
    joined = "\n\n".join([f"{i+1}. {txt}" for i, txt in enumerate(retrieved_texts)])
    prompt = f"""
    You are a legal assistant tasked with synthesizing general legal principles based on a collection of legal paragraphs associated with the title: "{title}".

    Do not summarize each paragraph individually. Do not reference any case numbers, laws, or directives.

    Your goal:
    1. Identify the overarching legal concept or procedural doctrine.
    2. Express it clearly and concisely (around 200 words).
    3. Mention briefly any consistent variations or exceptions observed, without detailing specific cases.

    Format the output as:
    ---
    General Concept:
    [Write here]

    Observed Variations:
    [Write here]
    ---

    ### Legal Paragraphs:
    {joined}
    """
    return prompt

# ---- 6. Funzione: chiama Ollama in modo sicuro ----
def generate_summary_with_ollama(prompt):
    try:
        result = subprocess.run(
            ["ollama", "run", "mistral"],
            input=prompt.encode("utf-8"),
            capture_output=True,
            timeout=300  # 5 minuti max
        )
        output = result.stdout.decode("utf-8").strip()
        if result.returncode != 0 or not output:
            raise RuntimeError("Ollama returned an error or empty output")
        return output
    except Exception as e:
        print(f"❌ Errore nella generazione con Ollama: {str(e)}")
        traceback.print_exc()
        return None

# ---- 7. MAIN LOOP migliorato ----
def process_all_titles(event_log, new_net):
    titles = [t.label for t in new_net.transitions if t.label is not None]
    summaries = {}

    for title in titles:
        texts = get_texts_for_title(event_log, title)
        
        if len(texts) < 3:
            print(f"⚠️ Titolo '{title}' ha pochi testi ({len(texts)}), salto...")
            continue

        print(f"📚 Indicizzazione: '{title}' ({len(texts)} testi)")
        index, _ = build_faiss_index(texts)

        retrieved = retrieve_top_k(texts, index, title, k=5)

        if len(retrieved) < 3:
            print(f"⚠️ Recuperati pochi testi rilevanti per '{title}', salto...")
            continue

        prompt = build_prompt(title, retrieved)
        summary = generate_summary_with_ollama(prompt)

        if summary:
            summaries[title] = summary
            print(f"✅ Sommario generato per '{title}':\n{summary}\n{'-'*60}")
        else:
            print(f"⚠️ Fallito generare il sommario per '{title}'.")

    return summaries



In [ ]:
# Esegui il processo completo
summaries = process_all_titles(event_log, new_net)

# Visualizza tutti i risultati
for title, summary in summaries.items():
    print(f"\n📝 Titolo: {title}\n{'='*60}")
    print(summary)
    print('\n' + '-'*60)


TF_IDF

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
import re
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

def extractive_summary_tfidf(paragraphs, top_k=3, max_words=50):
    # Unisci tutti i paragrafi
    full_text = " ".join(paragraphs)
    sentences = [s for s in sent_tokenize(full_text) if len(s.split()) <= max_words]


    if len(sentences) < top_k:
        return sentences  # troppo poche frasi da selezionare

    # TF-IDF sulle frasi
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(sentences)

    # Calcola punteggio medio per ogni frase
    scores = tfidf_matrix.sum(axis=1).flatten().tolist()[0]

    # Associa punteggi alle frasi e seleziona le top_k più rilevanti
    ranked_sentences = sorted(
        zip(sentences, scores),
        key=lambda x: x[1],
        reverse=True
    )

    summary_sentences = [sent for sent, _ in ranked_sentences[:top_k]]
    return summary_sentences

# Esempio di utilizzo
title = "admissibility"
paragraphs = get_texts_for_title(event_log, title)
summary_sentences = extractive_summary_tfidf(paragraphs, top_k=3)

print(f"📌 TF-IDF Summary per '{title}':\n")
for i, s in enumerate(summary_sentences):
    print(f"{i+1}. {s}")


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
import re
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

def extractive_summary_tfidf(paragraphs, top_k=3, max_words=50):
    # Unisci tutti i paragrafi
    full_text = " ".join(paragraphs)
    sentences = [s for s in sent_tokenize(full_text) if len(s.split()) <= max_words]


    if len(sentences) < top_k:
        return sentences  # troppo poche frasi da selezionare

    # TF-IDF sulle frasi
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(sentences)

    # Calcola punteggio medio per ogni frase
    scores = tfidf_matrix.sum(axis=1).flatten().tolist()[0]

    # Associa punteggi alle frasi e seleziona le top_k più rilevanti
    ranked_sentences = sorted(
        zip(sentences, scores),
        key=lambda x: x[1],
        reverse=True
    )

    summary_sentences = [sent for sent, _ in ranked_sentences[:top_k]]
    return summary_sentences

# Esempio di utilizzo
title = "legal context"
paragraphs = get_texts_for_title(event_log, title)
summary_sentences = extractive_summary_tfidf(paragraphs, top_k=3)

print(f"📌 TF-IDF Summary per '{title}':\n")
for i, s in enumerate(summary_sentences):
    print(f"{i+1}. {s}")


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
import re
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

def extractive_summary_tfidf(paragraphs, top_k=3, max_words=50):
    # Unisci tutti i paragrafi
    full_text = " ".join(paragraphs)
    sentences = [s for s in sent_tokenize(full_text) if len(s.split()) <= max_words]


    if len(sentences) < top_k:
        return sentences  # troppo poche frasi da selezionare

    # TF-IDF sulle frasi
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(sentences)

    # Calcola punteggio medio per ogni frase
    scores = tfidf_matrix.sum(axis=1).flatten().tolist()[0]

    # Associa punteggi alle frasi e seleziona le top_k più rilevanti
    ranked_sentences = sorted(
        zip(sentences, scores),
        key=lambda x: x[1],
        reverse=True
    )

    summary_sentences = [sent for sent, _ in ranked_sentences[:top_k]]
    return summary_sentences

# Esempio di utilizzo
title = "ground"
paragraphs = get_texts_for_title(event_log, title)
summary_sentences = extractive_summary_tfidf(paragraphs, top_k=3)

print(f"📌 TF-IDF Summary per '{title}':\n")
for i, s in enumerate(summary_sentences):
    print(f"{i+1}. {s}")


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
import re
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

def extractive_summary_tfidf(paragraphs, top_k=3, max_words=50):
    # Unisci tutti i paragrafi
    full_text = " ".join(paragraphs)
    sentences = [s for s in sent_tokenize(full_text) if len(s.split()) <= max_words]


    if len(sentences) < top_k:
        return sentences  # troppo poche frasi da selezionare

    # TF-IDF sulle frasi
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(sentences)

    # Calcola punteggio medio per ogni frase
    scores = tfidf_matrix.sum(axis=1).flatten().tolist()[0]

    # Associa punteggi alle frasi e seleziona le top_k più rilevanti
    ranked_sentences = sorted(
        zip(sentences, scores),
        key=lambda x: x[1],
        reverse=True
    )

    summary_sentences = [sent for sent, _ in ranked_sentences[:top_k]]
    return summary_sentences

# Esempio di utilizzo
title = "cost"
paragraphs = get_texts_for_title(event_log, title)
summary_sentences = extractive_summary_tfidf(paragraphs, top_k=3)

print(f"📌 TF-IDF Summary per '{title}':\n")
for i, s in enumerate(summary_sentences):
    print(f"{i+1}. {s}")


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
import re
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

def extractive_summary_tfidf(paragraphs, top_k=3, max_words=50):
    # Unisci tutti i paragrafi
    full_text = " ".join(paragraphs)
    sentences = [s for s in sent_tokenize(full_text) if len(s.split()) <= max_words]


    if len(sentences) < top_k:
        return sentences  # troppo poche frasi da selezionare

    # TF-IDF sulle frasi
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(sentences)

    # Calcola punteggio medio per ogni frase
    scores = tfidf_matrix.sum(axis=1).flatten().tolist()[0]

    # Associa punteggi alle frasi e seleziona le top_k più rilevanti
    ranked_sentences = sorted(
        zip(sentences, scores),
        key=lambda x: x[1],
        reverse=True
    )

    summary_sentences = [sent for sent, _ in ranked_sentences[:top_k]]
    return summary_sentences

# Esempio di utilizzo
title = "party"
paragraphs = get_texts_for_title(event_log, title)
summary_sentences = extractive_summary_tfidf(paragraphs, top_k=3)

print(f"📌 TF-IDF Summary per '{title}':\n")
for i, s in enumerate(summary_sentences):
    print(f"{i+1}. {s}")


Text_Rank

In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from nltk.tokenize import sent_tokenize  # Richiede nltk.download('punkt')

def extractive_summary_lexrank(paragraphs, top_k=3, max_sentences=50):
    # Unisci paragrafi in un testo unico
    full_text = " ".join(paragraphs)
    
    # Tokenizza in frasi e riduci a max_sentences
    all_sentences = sent_tokenize(full_text)
    reduced_text = " ".join(all_sentences[:max_sentences])
    
    # Inizializza parser e summarizer
    parser = PlaintextParser.from_string(reduced_text, Tokenizer("english"))  # Usa "italian" se serve
    summarizer = LexRankSummarizer()
    
    # Estrai le frasi più rilevanti (top_k)
    summary = summarizer(parser.document, top_k)
    
    return [str(sentence) for sentence in summary]

# === ESEMPIO USO ===
title = "admissibility"
paragraphs = get_texts_for_title(event_log, title)  # Supponendo questa funzione esista
summary_sentences = extractive_summary_lexrank(paragraphs, top_k=3, max_sentences=50)

print(f"📌 LexRank Summary per '{title}':\n")
for i, s in enumerate(summary_sentences):
    print(f"{i+1}. {s}")


In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from nltk.tokenize import sent_tokenize  # Richiede nltk.download('punkt')

def extractive_summary_lexrank(paragraphs, top_k=3, max_sentences=50):
    # Unisci paragrafi in un testo unico
    full_text = " ".join(paragraphs)
    
    # Tokenizza in frasi e riduci a max_sentences
    all_sentences = sent_tokenize(full_text)
    reduced_text = " ".join(all_sentences[:max_sentences])
    
    # Inizializza parser e summarizer
    parser = PlaintextParser.from_string(reduced_text, Tokenizer("english"))  # Usa "italian" se serve
    summarizer = LexRankSummarizer()
    
    # Estrai le frasi più rilevanti (top_k)
    summary = summarizer(parser.document, top_k)
    
    return [str(sentence) for sentence in summary]

# === ESEMPIO USO ===
title = "legal context"
paragraphs = get_texts_for_title(event_log, title)  # Supponendo questa funzione esista
summary_sentences = extractive_summary_lexrank(paragraphs, top_k=3, max_sentences=50)

print(f"📌 LexRank Summary per '{title}':\n")
for i, s in enumerate(summary_sentences):
    print(f"{i+1}. {s}")



In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from nltk.tokenize import sent_tokenize  # Richiede nltk.download('punkt')

def extractive_summary_lexrank(paragraphs, top_k=3, max_sentences=50):
    # Unisci paragrafi in un testo unico
    full_text = " ".join(paragraphs)
    
    # Tokenizza in frasi e riduci a max_sentences
    all_sentences = sent_tokenize(full_text)
    reduced_text = " ".join(all_sentences[:max_sentences])
    
    # Inizializza parser e summarizer
    parser = PlaintextParser.from_string(reduced_text, Tokenizer("english"))  # Usa "italian" se serve
    summarizer = LexRankSummarizer()
    
    # Estrai le frasi più rilevanti (top_k)
    summary = summarizer(parser.document, top_k)
    
    return [str(sentence) for sentence in summary]

# === ESEMPIO USO ===
title = "ground"
paragraphs = get_texts_for_title(event_log, title)  # Supponendo questa funzione esista
summary_sentences = extractive_summary_lexrank(paragraphs, top_k=3, max_sentences=50)

print(f"📌 LexRank Summary per '{title}':\n")
for i, s in enumerate(summary_sentences):
    print(f"{i+1}. {s}")



In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from nltk.tokenize import sent_tokenize  # Richiede nltk.download('punkt')

def extractive_summary_lexrank(paragraphs, top_k=3, max_sentences=50):
    # Unisci paragrafi in un testo unico
    full_text = " ".join(paragraphs)
    
    # Tokenizza in frasi e riduci a max_sentences
    all_sentences = sent_tokenize(full_text)
    reduced_text = " ".join(all_sentences[:max_sentences])
    
    # Inizializza parser e summarizer
    parser = PlaintextParser.from_string(reduced_text, Tokenizer("english"))  # Usa "italian" se serve
    summarizer = LexRankSummarizer()
    
    # Estrai le frasi più rilevanti (top_k)
    summary = summarizer(parser.document, top_k)
    
    return [str(sentence) for sentence in summary]

# === ESEMPIO USO ===
title = "cost"
paragraphs = get_texts_for_title(event_log, title)  # Supponendo questa funzione esista
summary_sentences = extractive_summary_lexrank(paragraphs, top_k=3, max_sentences=50)

print(f"📌 LexRank Summary per '{title}':\n")
for i, s in enumerate(summary_sentences):
    print(f"{i+1}. {s}")



In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from nltk.tokenize import sent_tokenize  # Richiede nltk.download('punkt')

def extractive_summary_lexrank(paragraphs, top_k=3, max_sentences=50):
    # Unisci paragrafi in un testo unico
    full_text = " ".join(paragraphs)
    
    # Tokenizza in frasi e riduci a max_sentences
    all_sentences = sent_tokenize(full_text)
    reduced_text = " ".join(all_sentences[:max_sentences])
    
    # Inizializza parser e summarizer
    parser = PlaintextParser.from_string(reduced_text, Tokenizer("english"))  # Usa "italian" se serve
    summarizer = LexRankSummarizer()
    
    # Estrai le frasi più rilevanti (top_k)
    summary = summarizer(parser.document, top_k)
    
    return [str(sentence) for sentence in summary]

# === ESEMPIO USO ===
title = "party"
paragraphs = get_texts_for_title(event_log, title)  # Supponendo questa funzione esista
summary_sentences = extractive_summary_lexrank(paragraphs, top_k=3, max_sentences=50)

print(f"📌 LexRank Summary per '{title}':\n")
for i, s in enumerate(summary_sentences):
    print(f"{i+1}. {s}")



In [ ]:
import pickle

# Percorso del file .pkl
file_path = 'legal_summaries.pkl'

# Apri e carica il contenuto
with open(file_path, 'rb') as file:
    dati = pickle.load(file)

# Ora puoi usare i dati
print(dati)
